In [123]:
# CÉLULA 1: Importando as bibliotecas
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [124]:
# CÉLULA 2: Carregando os dados
df = pd.read_csv('limites_de_credito_1000_registros.csv')

print("Visualizando os primeiros registros da base:\n")
print(df.head())

Visualizando os primeiros registros da base:

   Idade  Salario  Score_Credito  Limite_Aprovado
0     68     5133            533          2797.23
1     27     4621            706          4658.81
2     30    22716            786         11425.50
3     68    15653            940         10384.01
4     24     4957            433          2055.79


In [125]:
# CÉLULA 3: Separando entradas e saída
X = df[['Idade', 'Salario', "Score_Credito"]]
y = df ['Limite_Aprovado']

print("Variáveis de entrada:")
print(X.head())

print("Variável alvo:")
print(y.head())

Variáveis de entrada:
   Idade  Salario  Score_Credito
0     68     5133            533
1     27     4621            706
2     30    22716            786
3     68    15653            940
4     24     4957            433
Variável alvo:
0     2797.23
1     4658.81
2    11425.50
3    10384.01
4     2055.79
Name: Limite_Aprovado, dtype: float64


In [126]:
# CÉLULA 4: Separando treino e teste
X_treino, X_teste, y_treino, y_teste = \
train_test_split(X,y, test_size=0.2, random_state=42)

print("Tamanho do conjunto de treino:", X_treino.shape)
print("Tamanho do conjunto de teste:", X_teste.shape)

Tamanho do conjunto de treino: (800, 3)
Tamanho do conjunto de teste: (200, 3)


In [127]:
# CÉLULA 5: Normalizando as entradas e o alvo
normalizador_x = StandardScaler()
X_treino_norm = normalizador_x.fit_transform(X_treino)
X_teste_norm = normalizador_x.transform(X_teste)

normalizador_y = StandardScaler()
y_treino_norm = normalizador_y.fit_transform(y_treino.values.reshape(-1,1))
y_teste_norm = normalizador_y.transform(y_teste.values.reshape(-1,1))

print("Normalização concluída!")

Normalização concluída!


In [128]:
# CÉLULA 6: Construindo a rede neural de regressão
tf.random.set_seed(42)
modelo_limite = keras.Sequential([
    keras.Input(shape=(3,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1)
])

modelo_limite.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

modelo_limite.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_27 (Dense)                │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209 (836.00 B)

 Trainable params: 209 (836.00 B)

 Non-trainable params: 0 (0.00 B)

In [129]:
# CÉLULA 7: Treinando o modelo

print("Treinando o avaliador de limites ...\n")

historico = modelo_limite.fit(
    X_treino_norm,
    y_treino_norm,
    epochs=100,
    validation_split=0.2,
    verbose=0
)

print("\nTreinamento concluído")

Treinando o avaliador de limites ...


Treinamento concluído


In [130]:
# CÉLULA 8: Avaliando o modelo

perda, mae = modelo_limite.evaluate(X_teste_norm, y_teste_norm)

print(f"Perda no teste, em escala normalizada: {perda:.4f}")
print(f"Mae no teste, em escala normalizada: {mae:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0718 - mae: 0.2139  
Perda no teste, em escala normalizada: 0.0718
Mae no teste, em escala normalizada: 0.2139


In [131]:
# CÉLULA 9: Convertendo previsões para Reais

previsoes_norm = modelo_limite.predict(X_teste_norm)
previsoes_reais = normalizador_y.inverse_transform(previsoes_norm)
y_teste_reais = y_teste.values.reshape(-1,1)
mae_reais = mean_absolute_error(y_teste_reais,previsoes_reais)
rmse_reais = np.sqrt(mean_squared_error(y_teste_reais, previsoes_reais))

print(f"Erro médio absoluto em Rais: R$ {mae_reais:.2f}")
print(f"Raiz do erro quadrático médio em Reais: R$ {rmse_reais:.2f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Erro médio absoluto em Rais: R$ 578.12
Raiz do erro quadrático médio em Reais: R$ 724.06


In [132]:
# CÉLULA 10: Testando com um novo cliente

idade = int(input("Informe a sua idade: "))
salario = float(input("Informe o seu salário: "))
score = int(input("Informe o seu score de crédito: "))

cliente_novo = pd.DataFrame({
    'Idade': [idade],
    'Salario': [salario],
    'Score_Credito': [score]
})

cliente_norm = normalizador_x.transform(cliente_novo)
limite_previsto_norm = modelo_limite.predict(cliente_norm)
limite_real = normalizador_y.inverse_transform(limite_previsto_norm)

print(f"Limite de crédito sugerido: R$ {limite_real[0][0]:.2f}")

Informe a sua idade: 27
Informe o seu salário: 5000
Informe o seu score de crédito: 1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
Limite de crédito sugerido: R$ 6218.75
